In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

sudan = pd.read_csv("evaluation/sudan_results.csv")
sudan_hist = pd.read_csv("evaluation/sudan_results_historical.csv") # Before the collapse
ethiopia = pd.read_csv("evaluation/ethiopia_results.csv")

print(f"Sudan (current): {len(sudan)} runs")
print(f"Sudan (historical, includes k=0.25): {len(sudan_hist)} runs")
print(f"Ethiopia: {len(ethiopia)} runs")

Sudan (current): 160 runs
Sudan (historical, includes k=0.25): 336 runs
Ethiopia: 32 runs


### K - escalation threshold
`k` controls the esclation threshold, a lower k = a looser threshold. 
Raw AUPR favours the lowest `k`, but that is misleading in terms of conflict prediction, the model was catching 100% of true positives by deafult rather than skill.

After this was discovered, k was set to 1 for subsequent runs.

In [5]:
collapsed = sudan_hist[sudan_hist["onset_recall_class1"] == 1.0]
baseline_by_k = collapsed.groupby("k")["onset_precision_class1"].median()
collapse_rate = sudan_hist.groupby("k")["onset_recall_class1"].apply(lambda x: (x == 1.0).mean())

k_summary = pd.DataFrame({
    "mean_onset_aupr": sudan_hist.groupby("k")["onset_aupr"].mean(),
    "max_onset_aupr": sudan_hist.groupby("k")["onset_aupr"].max(),
    "baseline_prevalence": baseline_by_k,
    "collapse_rate": collapse_rate,
})
k_summary.round(3)

,mean_onset_aupr,max_onset_aupr,baseline_prevalence,collapse_rate
k,,,,
0.25,0.440,0.526,0.421,0.750
0.50,0.368,0.458,0.370,0.641
1.00,0.338,0.421,0.306,0.281


In [7]:
ks = k_summary.index.astype(str)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Positive-class prevalence by k", "Threshold-collapse rate (%)"))

fig.add_trace(
    go.Bar(x=ks, y=k_summary["baseline_prevalence"], marker_color="#898781", name="Baseline prevalence"),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(x=ks, y=k_summary["collapse_rate"] * 100, marker_color="#c0392b", name="Collapse rate"),
    row=1, col=2,
)

fig.update_xaxes(title_text="k", row=1, col=1)
fig.update_xaxes(title_text="k", row=1, col=2)
fig.update_yaxes(title_text="% of runs with recall=1.0", row=1, col=2)

fig.update_layout(
    showlegend=False, width=900, height=400,
    plot_bgcolor="white", paper_bgcolor="white",
    title_text="A lower escalation threshold (k) had the highest AUPR but led to a high collapse rate",
)
fig.show()